
# The Fundamental Metallicity Relation: M*-Z-SFR three-body interaction

The Mannucci+2010 fundamental metallicity relation (FMR) describes how
a galaxy's gas-phase metallicity (Z) depends not only on its stellar mass
(M*) but also on its star formation rate (SFR). This three-parameter
relation is a *schematic* demonstration of the physical interplay between
assembly, star formation, and chemical enrichment.

The left panel shows the classic M* vs. Z scatter colored by SFR. The
right panel projects to M* - μ (where μ = log M* - 0.32 log SFR) versus Z,
revealing that the FMR scatter tightens along this projection—a signature
of the outflow-driven "mass-loading" scenario.

**Note:** The metallicity grid shown here is purely schematic, generated
from an analytic function of (M*, sSFR). Real galaxies follow the FMR
with ~0.1 dex scatter, dominated by dust, ISM geometry, and starburst
history.

References:
- Mannucci et al. 2010, MNRAS, 408, 2115 (the FMR definition)
- Lara-López et al. 2010, A&A, 521, L53 (confirmation with SDSS)


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import matplotlib.pyplot as plt
import numpy as np

from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

# ==============================================================================
# Schematic FMR metallicity function: Z as function of M* and sSFR
# ==============================================================================


def schematic_metallicity(log_m_star, log_ssfr):
    r"""
    Schematic metallicity following Mannucci+2010 FMR ansatz.

    Parameters
    ----------
    log_m_star : ndarray
        Log10(M_star / M_sun)
    log_ssfr : ndarray
        Log10(sSFR / yr^-1)

    Returns
    -------
    log_z_over_zsun : ndarray
        Log10(Z / Z_sun), where Z_sun = 0.02 in linear units
    """
    # Base: mass-metallicity relation at fixed reference sSFR
    base = -0.5 + 0.3 * (log_m_star - 10.0)

    # SFR-driven offset: more SFR pushes Z down (outflow-driven)
    # Normalization: log_ssfr ≈ -10.5 for main sequence
    sfr_offset = -0.15 * (log_ssfr + 10.0)

    # Small scatter: 0.05 dex per-point, static for reproducibility
    np.random.seed(42)
    scatter = np.random.normal(0, 0.05, size=np.asarray(log_m_star).shape)

    return base + sfr_offset + scatter


# ==============================================================================
# Generate grid: (M*, sSFR) → (M*, Z)
# ==============================================================================

log_m_star_vals = np.linspace(9.0, 11.0, 6)  # 9, 9.4, 9.8, 10.2, 10.6, 11.0
log_ssfr_vals = np.linspace(-11.0, -9.0, 5)  # -11, -10.5, -10, -9.5, -9

# Create 2D mesh
log_m_grid, log_ssfr_grid = np.meshgrid(log_m_star_vals, log_ssfr_vals)
log_m_flat = log_m_grid.ravel()
log_ssfr_flat = log_ssfr_grid.ravel()

# Compute metallicity
log_z_flat = schematic_metallicity(log_m_flat, log_ssfr_flat)

# Extract SFR for coloring (log sSFR → linear sSFR for visual scaling)
ssfr_linear = 10.0**log_ssfr_flat

# ==============================================================================
# Left panel: M* vs Z colored by SFR
# ==============================================================================

fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(12.0, 5.0))

# Scatter with SFR as color dimension
sc_left = ax_left.scatter(
    log_m_flat,
    log_z_flat,
    c=log_ssfr_flat,
    cmap="plasma",
    s=80,
    lw=0.8,
    edgecolor="white",
    alpha=0.85,
    vmin=-11.0,
    vmax=-9.0,
)

ax_left.set_xlabel(r"$\log\,(M_\star\,/\,M_\odot)$", fontsize=11)
ax_left.set_ylabel(r"$\log\,(Z\,/\,Z_\odot)$", fontsize=11)
ax_left.set_ylim(-1.5, 0.5)
ax_left.set_xlim(8.8, 11.2)
ax_left.grid(True, alpha=0.2, linestyle=":")

cb_left = fig.colorbar(sc_left, ax=ax_left, pad=0.02)
cb_left.set_label(r"$\log\,(\mathrm{sSFR}\,/\,\mathrm{yr}^{-1})$", fontsize=10)

# ==============================================================================
# Right panel: μ = log M* - 0.32 log sSFR vs Z
# ==============================================================================

# The "Fundamental Plane" projection (Mannucci+2010 ansatz)
mu = log_m_flat - 0.32 * log_ssfr_flat

sc_right = ax_right.scatter(
    mu,
    log_z_flat,
    c=log_ssfr_flat,
    cmap="plasma",
    s=80,
    lw=0.8,
    edgecolor="white",
    alpha=0.85,
    vmin=-11.0,
    vmax=-9.0,
)

ax_right.set_xlabel(
    r"$\mu\,=\,\log\,M_\star - 0.32\,\log\,\mathrm{sSFR}$",
    fontsize=11,
)
ax_right.set_ylabel(r"$\log\,(Z\,/\,Z_\odot)$", fontsize=11)
ax_right.set_ylim(-1.5, 0.5)
ax_right.grid(True, alpha=0.2, linestyle=":")

# Fit a tight line to show the FMR tightens along μ
z_sorted = log_z_flat[np.argsort(mu)]
mu_sorted = np.sort(mu)
p_fit = np.polyfit(mu_sorted, z_sorted, 1)
mu_line = np.linspace(mu.min() - 0.2, mu.max() + 0.2, 50)
z_line = np.polyval(p_fit, mu_line)
ax_right.plot(
    mu_line,
    z_line,
    color="gray",
    lw=1.2,
    ls="--",
    alpha=0.6,
    label=f"Best fit (slope={p_fit[0]:.2f})",
)
ax_right.legend(fontsize=9, loc="lower right", frameon=True, fancybox=True)
ax_right.set_xlim(mu.min() - 0.3, mu.max() + 0.3)

fig.tight_layout()

plt.savefig("plot_usecase_fundamental_metallicity_relation.png", dpi=150, bbox_inches="tight")